In [2]:
import os
import pandas as pd
import glob

def process_tcga_clinical_data(base_dir):
    """
    Process TCGA clinical data and integrate clinical data for all cancer types.
    
    Parameters:
    base_dir: The path to the main directory containing all downloaded folders
    
    Returns:
    A dictionary containing the merged data, with keys as data types (e.g., clinical_patient, clinical_drug, etc.).
    """
    # Store data frames of different types.
    data_dict = {}
    
    # Iterate through all folders.
    for folder in glob.glob(os.path.join(base_dir, "*")):
        if not os.path.isdir(folder):
            continue
            
        # Retrieve the cancer type (from the folder name).
        cancer_type = os.path.basename(folder)
        
        # Iterate through all biotab files in the folder.
        for filepath in glob.glob(os.path.join(folder, "*.txt")):
            filename = os.path.basename(filepath)
            
            try:
                
                df = pd.read_csv(filepath, sep='\t', comment='#')
                
                df['cancer_type'] = cancer_type
                
                if 'clinical_patient' in filename:
                    key = 'clinical_patient'
                elif 'clinical_drug' in filename:
                    key = 'clinical_drug'
                elif 'clinical_follow_up' in filename:
                    key = 'clinical_follow_up'
                elif 'biospecimen' in filename:
                    key = 'biospecimen'
                else:
                    key = 'other'
                
                # Add the data to the corresponding dictionary entry.
                if key in data_dict:
                    data_dict[key].append(df)
                else:
                    data_dict[key] = [df]
                    
            except Exception as e:
                print(f"An error occurred while processing the file {filepath}: {str(e)}")
    
    # Merge the data frames for each type.
    for key in data_dict:
        try:
            data_dict[key] = pd.concat(data_dict[key], ignore_index=True)
        except Exception as e:
            print(f"An error occurred while merging the {key} data: {str(e)}")
    
    return data_dict

def save_processed_data(data_dict, output_dir):
    """
    Save the processed data to the specified directory.
    
    Parameters:
    data_dict: A dictionary containing the processed data.
    output_dir: The path to the output directory.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    for key, df in data_dict.items():
        output_path = os.path.join(output_dir, f"{key}_merged.csv")
        try:
            df.to_csv(output_path, index=False)
            print(f"{key} data has been saved to: {output_path}")
        except Exception as e:
            print(f"An error occurred while saving {key} data: {str(e)}")

def analyze_data_summary(data_dict):
    """
    Generate a data summary report.
    
    Parameters:
    data_dict: A dictionary containing the processed data.
    """
    summary = {}
    for key, df in data_dict.items():
        summary[key] = {
            'Number of samples': len(df),
            'Number of cancer types': len(df['cancer_type'].unique()),
            'Number of columns': df.shape[1],
            'Column names': df.columns.tolist()
        }
    return summary

In [3]:

base_dir = "tcga_biotab/gdc_download_20241114_034159.633379"  # Replace with your data directory.
output_dir = "tcga_result"  # Replace with the directory where you want to save the results.

# Process the data.
data_dict = process_tcga_clinical_data(base_dir)

# Save the processed data.
save_processed_data(data_dict, output_dir)

# Retrieve the data summary.
summary = analyze_data_summary(data_dict)

处理文件时出错 tcga_biotab/gdc_download_20241114_034159.633379\7ff01c94-2d42-4686-89e6-cdf2e8e4cc7b\nationwidechildrens.org_clinical_omf_v4.0_blca.txt: 'utf-8' codec can't decode byte 0x96 in position 23933: invalid start byte
处理文件时出错 tcga_biotab/gdc_download_20241114_034159.633379\f1cb9e8d-9d99-45d8-b189-afaf772dd477\nationwidechildrens.org_clinical_omf_v4.0_lusc.txt: 'utf-8' codec can't decode byte 0x96 in position 32942: invalid start byte
已保存other数据到: tcga_result\other_merged.csv
已保存clinical_patient数据到: tcga_result\clinical_patient_merged.csv
已保存clinical_follow_up数据到: tcga_result\clinical_follow_up_merged.csv
已保存clinical_drug数据到: tcga_result\clinical_drug_merged.csv
